# 配电网可规划域：逐线路选型模型审核

本轮默认运行 **case33 的四条候选线路、两个型号**，核对紧凑 MILP/MISOCP、含规划变量的联合割与原 16 方案枚举。模型及联合割推导见 [docs/compact_planning.md](docs/compact_planning.md)。

Network 唯一保存物理数据和线路选项；model.py 建模；vertify.py 核对割和计算汇总；本 Notebook 组织求解与实验。紧凑求解器不读取完整方案表；本轮枚举仅作为独立对照。

COMPACT_REVIEW=False 可运行下方保留的小规模区域/AC 基准流程。此次审核不扩展到全网升级，也不把若干射线点的凸包称为整个规划域。


In [1]:
from importlib import import_module
from pathlib import Path
from time import perf_counter
from hashlib import sha256
import json
import nbformat
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from scipy.spatial import HalfspaceIntersection
from threadpoolctl import threadpool_limits
from IPython.display import display, IFrame

from model import MP1, MP2, LinearSP, SOCPSP, ACPowerFlow, BranchEquations, add_cut
from region import (simplex, halfspaces, polytope_vertices, update_outer,
                    add_certificate, ResidualSearch)
from vertify import BenchmarkResult, region_membership, validate_power_flow, METHODS, METHOD_NAMES
from plot import save_method_comparison, save_replay

from model import PlanningModel, PlanningSP, dispatch_support
from vertify import verify_joint_cuts, compact_review_summary


In [2]:
COMPACT_REVIEW = True             # 本轮先核对四条候选线路；False 运行原小规模区域基准
CASE = "case33bw"                 # 或 "four_bus_five_corridor"
PLANNING = True                    # False：固定最低费用方案
BUDGETS = (0., 1., 2., np.inf)      # 四节点用 (20000., 40000., 60000., np.inf)
DIVISIONS = 128                    # 每轴网格数；只影响事后评价
RADIAL_TOLERANCE = .002            # SOCP 连续内外域的径向停止精度
RECOMPUTE = True                   # False：明确读取本实验的 result.npz
OUTPUT = Path("results")/CASE/("planning" if PLANNING else "fixed")


**网架与对照数据。** 四条候选支路 A、B、C、D 的两个型号分别为保持原状与同走廊并联一回，费用及 R/X 与原 16 方案完全相同。紧凑模型有 8 个逐线路型号变量，每条线路满足 $\sum_k x_{e,k}=1$。

以下 16 方案表只用于核对结果。旧小规模区域基准仍使用完整方案选择变量；它不参与新的紧凑主问题和联合割。

In [3]:
network = import_module(f"Network.{CASE}").network  # 网架、背景负荷、可选建设项目只从选定算例读取。
designs = network.designs if PLANNING else network.designs[:1]  # 方案表已按费用排序；固定网架模式只取最低费用方案。
budgets = BUDGETS if PLANNING else (np.inf,)  # 固定方案无需预算筛选，规划模式逐档比较允许方案的并集。
print(f"{CASE}: {len(designs)} 个建设方案；独立负荷节点 {designs[0].load_nodes}")
display(pd.DataFrame([dict(方案=i, 建设变量=tuple(d.x), 费用=d.cost)
                      for i,d in enumerate(designs)]))


case33bw: 16 个建设方案；独立负荷节点 (18, 25, 33)


,方案,建设变量,费用
0,0,"(0, 0, 0, 0)",0.0
1,1,"(0, 0, 0, 1)",1.0
2,2,"(0, 0, 1, 0)",1.0
3,3,"(0, 1, 0, 0)",1.0
4,4,"(0, 0, 1, 1)",2.0
5,5,"(0, 1, 0, 1)",2.0
6,6,"(0, 1, 1, 0)",2.0
7,7,"(1, 0, 0, 0)",2.0
8,8,"(0, 1, 1, 1)",3.0
9,9,"(1, 0, 0, 1)",3.0


**紧凑模型与联合割审核。** 主问题直接选择线路型号；固定 $x,p$ 后解连续 phase I，割为
$$a+b^\top p+d^\top x\ge0.$$
这里的循环没有完整方案索引，也没有人为迭代次数作为收敛依据。算法只接受经过原约束核验的可行状态，或具有对偶锥与有限界补偿的有效分离割。SOCP 证书仍是松弛模型的证书，不等同于 AC 等式认证。

负荷最大化查询用 MP 上界与相差 0.001 kW 的认证内点停止，记录两者；固定负荷的最小投资查询不移动负荷坐标。

In [4]:
def joint_benders(network, method, *, power=None, budget=np.inf, direction=None, cuts=(), radial_gap_kw=1e-3):
    """一次非枚举规划查询；跨查询复用的每条割均包含 p 和逐线路选型 x。"""
    problem = PlanningModel(network,method,power=power,budget=budget,direction=direction,cuts_only=True)
    oracle = PlanningSP(problem.equations)
    generated, calls = [], 0
    with problem.model:  # 主问题只在本次查询内存活；退出时释放求解资源。
        for cut in cuts:
            problem.add_cut(cut)
        while True:
            candidate = problem.solve()  # MP 是原问题的外松弛，最优值提供投资下界或负荷上界。
            if candidate is None:
                return None,generated,calls  # 外松弛已不可行，原规划问题也不可行。
            check_power = candidate['p']
            if power is None:
                check_power = check_power*(1.-radial_gap_kw/check_power.sum())  # 内侧点与 MP 负荷上界只差指定 kW 间隙，避免强行认证浮点边界。
            result = oracle.solve(candidate['x'],check_power)
            calls += 1
            if result['cut'] is None:
                candidate['state'] = result['state']
                candidate['p'] = check_power
                if power is None:
                    candidate['objective'] = check_power.sum()  # 返回认证内点；bound 保留 MP 的全局上界。
                return candidate,generated,calls
            generated.append(result['cut'])
            problem.add_cut(result['cut'])  # a+bᵀp+dᵀx≥0，同时限制负荷和建设组合。


In [5]:
def review_compact(network, budgets, radial_gap_kw=1e-3):
    """只核对当前四条候选线路；枚举参考和紧凑求解分开计时、使用不同潮流表示。"""
    startup = gp.Model(); startup.dispose()  # 公共许可证启动在两种方法的计时之外。
    rng = np.random.default_rng(20260920)
    directions = np.vstack([np.eye(3),[1.,1.,1.],[1.,6.,1.],[2.,3.,1.],
                            rng.dirichlet([1.,3.,1.],size=6)])
    directions /= directions.sum(axis=1,keepdims=True)  # 射线系数和为 1，半径就是三个独立负荷之和，kW。
    plans = network.designs  # 枚举只出现在审核流程；PlanningModel/PlanningSP 不调用此属性。
    costs = np.array([d.cost for d in plans])
    record = dict(budgets=[None if np.isinf(b) else b for b in budgets],directions=directions,
                  design_choices=[d.x for d in plans],design_costs=costs,radial_gap_kw=radial_gap_kw,models={})
    for method in ('linear','socp'):
        start = perf_counter()
        radii = np.array([[dispatch_support(d,method,np.ones(3),direction=a)['value']
                          for a in directions] for d in plans])  # 16 个独立消元模型逐方案求边界。
        record['models'][method] = dict(enumerated_radii=radii,
                                        seconds=dict(enumeration_boundary_seconds=perf_counter()-start))
    bounds = np.max(record['models']['linear']['enumerated_radii'][:,:3],axis=0)
    points = [rng.random((32,3))*bounds]  # 均匀箱内样本，含域内及域外点。
    for data in record['models'].values():
        for budget in budgets:
            boundary = np.max(data['enumerated_radii'][costs<=budget],axis=0)
            for scale in (.999,1.001):  # 显式检查 LP/SOCP 各预算边界的两侧，不只选容易的内部点。
                points.append(scale*boundary[:6,None]*directions[:6])
    points = np.unique(np.round(np.vstack(points),8),axis=0)
    record['points'] = points
    for method,data in record['models'].items():
        start = perf_counter()
        minimum = np.full(len(points),np.inf)
        for design in plans:
            oracle = LinearSP(design) if method=='linear' else SOCPSP(design)
            for i,power in enumerate(points):
                eta,_,_,_ = oracle.solve(power)
                if eta<=1e-8:
                    minimum[i] = min(minimum[i],design.cost)  # 穷尽 16 方案得到参考最小投资。
            oracle.close()
        data['enumerated_cost'] = minimum
        data['seconds']['enumeration_point_seconds'] = perf_counter()-start
        start = perf_counter()
        compact_cost, choices, radii = [], [], []
        for power in points:
            problem = PlanningModel(network,method,power=power)
            with problem.model:
                answer = problem.solve()
            compact_cost.append(np.inf if answer is None else answer['objective'])
            choices.append(None if answer is None else problem.equations.choice(answer['x']).tolist())
        data['compact_cost'],data['compact_choices'] = compact_cost,choices
        data['seconds']['compact_point_seconds'] = perf_counter()-start
        start = perf_counter()
        for budget in budgets:
            values = []
            for direction in directions:
                problem = PlanningModel(network,method,budget=budget,direction=direction)
                with problem.model:
                    answer = problem.solve()
                values.append(answer['objective'])
            radii.append(values)
        data['compact_radii'] = radii
        data['seconds']['compact_boundary_seconds'] = perf_counter()-start
        print(f"{method}: {len(points)} 个投资查询、{len(budgets)*len(directions)} 个边界查询完成",flush=True)
        start = perf_counter()
        cuts, queries = [], []
        for i in np.linspace(0,len(points)-1,20,dtype=int):
            answer,new,calls = joint_benders(network,method,power=points[i],cuts=cuts)
            cuts.extend(new)  # 全局有效割跨负荷点和预算复用，不保存完整方案表。
            queries.append(dict(kind='point',index=int(i),objective=np.inf if answer is None else answer['objective'],calls=calls))
        for j,budget in enumerate(budgets):
            for i in (0,1,2,4):
                answer,new,calls = joint_benders(network,method,budget=budget,direction=directions[i],cuts=cuts,radial_gap_kw=radial_gap_kw)
                cuts.extend(new)
                queries.append(dict(kind='ray',budget=j,index=i,objective=answer['objective'],bound=answer['bound'],calls=calls))
        data['cuts'],data['joint_queries'] = cuts,queries
        data['seconds']['joint_query_seconds'] = perf_counter()-start
        # 在全部 16 个连续方案域上优化检查每一条割，不只看其是否排除了生成点。
        data['cut_minimum'] = verify_joint_cuts(network,method,cuts)
        print(f"{method}: 联合割查询 {len(queries)} 次，全部 {len(cuts)} 条割完成 16 方案连续域核验",flush=True)
    record['hashes'] = {p:sha256(Path(p).read_bytes()).hexdigest()
                        for p in ('model.py','vertify.py',*network.sources)}
    record['hashes']['main.ipynb'] = sha256('\n'.join(c.source for c in nbformat.read('main.ipynb',4).cells).encode()).hexdigest()
    return record


**以下为原小规模枚举区域基准，只有 COMPACT_REVIEW=False 时才执行。**

**2．主问题—子问题迭代。** MP1 求指定负荷的最低建设费，MP2 求预算内最大总负荷。主问题给出方案与负荷，LP SP 返回违反量和对偶割；可行后停止本次查询。每条割只在产生该割的建设方案下生效。

下面是实际使用的循环；模型中没有另一个实验调度器。

In [6]:
def benders(problem, oracles, mode, target):  # 一次 MP/SP 查询；每轮记录立即交给外层维护候选域和证书。
    master, selection, power = problem  # 主问题的方案选择 z、独立负荷 p；潮流由固定方案 SP 检查。
    try:
        for _ in range(10000):  # 迭代上限仅用于发现未收敛；超限不会当作认证完成。
            master.optimize()  # 求当前全部历史条件割约束下的全局最优主问题。
            if master.Status == GRB.INFEASIBLE:  # 外松弛已不可行，真实 LP 规划问题也不可能可行。
                yield dict(mode=mode, target=target, design=None, p=None, eta=None, cut=None)  # 记录不可行证明，外层据此清空相应预算的候选域。
                return
            if master.Status != GRB.OPTIMAL:  # 非最优的目标值不能作为全域边界。
                raise RuntimeError(f"{mode}: master status {master.Status}")
            i = int(np.argmax([v.X for v in selection.values()]))  # 由 one-hot 的 z 提取被选中的完整方案编号。
            p = np.array([v.X for v in power.values()])  # 提取 MP 给出的三个节点负荷，单位 kW。
            eta, cut, _, _ = oracles[i].solve(p)  # 仅检查当前选中方案；这里使用 LP SP。
            if cut is not None:  # eta 非零且获得有效分离割时继续收紧主问题。
                add_cut(master, selection, power, i, cut)  # 条件割仅在 z_i=1 时生效，其他方案的负荷不受此割约束。
            yield dict(mode=mode, target=target, design=i, p=p, eta=eta, cut=cut)  # 记录一次真实 MP/SP 计算，不复制建设向量和费用。
            if cut is None:  # LP SP 已在容差内认证此点，当前 MP 查询结束。
                return
        raise RuntimeError("Benders iteration limit reached")
    finally:
        master.dispose()  # 释放本次主问题；后续查询由 history 恢复有效割。


def linear_region(plans, budget):  # 先查询投资—总负荷前沿，再补齐完整三维 LP 并集。
    oracles = [LinearSP(d) for d in plans]  # 每个合法离散建设方案构建一个可复用 LP SP。
    costs = np.array([d.cost for d in plans])  # 方案费用只读取 Network 的方案表。
    outer = [simplex(d) for d in plans]  # 每个方案维护独立外域，不能把不同方案直接混成一个凸包。
    certified, history = {}, []  # 按方案存内域；history 是本次 LP 查询的唯一原始记录。
    limit = budget  # 从本次最高预算开始向更低投资档位推进。
    try:
        # MP2 -> 总量 MP1，逐个跳到更低的实际建设费用，不使用整数费用粒度。
        while True:
            for row in benders(MP2(plans, limit, history), oracles, 'MP2', limit):  # MP2 求当前预算下可承载的最大独立节点总负荷。
                history.append(row)  # 保存本轮查询点、方案、违反量及割。
                outer = update_outer(costs, outer, row)  # 应用方案割；认证的 MP2 最优总量也可收紧预算内所有外域。
                add_certificate(certified, row)  # 仅把通过 LP SP 的点加入其所属方案的内域。
            if row['eta'] is None:  # 检查本次 MP 查询是否已证明不可行。
                break
            target = float(sum(row['p']))  # 把已认证 MP2 最优点的总负荷作为下一次 MP1 总量下限。
            for row in benders(MP1(plans, target, history, total=True), oracles, 'MP1-total', target):  # MP1 在保持该总负荷的条件下寻找最低建设费用。
                history.append(row)  # 保存本轮查询点、方案、违反量及割。
                outer = update_outer(costs, outer, row)  # 应用方案割；认证的 MP2 最优总量也可收紧预算内所有外域。
                add_certificate(certified, row)  # 仅把通过 LP SP 的点加入其所属方案的内域。
            if row['eta'] is None:  # 检查本次 MP 查询是否已证明不可行。
                raise RuntimeError("A certified MP2 point was lost in MP1")
            cheaper = costs[costs < costs[row['design']]-1e-8]  # 严格更便宜的实际投资档位；1e-8 仅排除费用浮点误差。
            if not len(cheaper):  # 已到最低实际费用，不再产生更低预算查询。
                break
            limit = float(cheaper.max())  # 跳到下一个较低费用档位；零投资与非整数费用同样适用。
        # 补齐费用前沿没有覆盖的三维区域；不同方案始终保留为并集。
        search = ResidualSearch(outer)  # 从尚未被认证并集覆盖的候选残块中选顶点。
        for _ in range(10000):  # 迭代上限仅用于发现未收敛；超限不会当作认证完成。
            candidate = search.next(outer, certified)  # 已认证并集可跳过整块，但块内凸组合必须来自同一个方案。
            if candidate is None:  # 所有残块均已被割删除或由认证并集覆盖，LP 构域完成。
                break
            i, p = candidate  # 待检查顶点及其所属候选方案。
            eta, cut, _, _ = oracles[i].solve(p)  # 仅检查当前选中方案；这里使用 LP SP。
            row = dict(mode='vertex', target=None, design=int(i), p=p, eta=eta, cut=cut)  # 记录补齐阶段的一次真实顶点 SP 查询。
            history.append(row)  # 保存本轮查询点、方案、违反量及割。
            outer = update_outer(costs, outer, row)  # 应用方案割；认证的 MP2 最优总量也可收紧预算内所有外域。
            add_certificate(certified, row)  # 仅把通过 LP SP 的点加入其所属方案的内域。
        else:
            raise RuntimeError("Linear region certification did not finish")
        records = [dict(design=i, initial=simplex(d), outer=outer[i],  # 保存各方案的初始域、最终外域和已认证内域。
                        inner=certified.get(i, np.zeros((1,3))),
                        history=[r for r in history if r['design']==i], calls=oracles[i].calls)  # 按方案归属查询历史，用于核查割及回放。
                   for i,d in enumerate(plans)]
        return records
    finally:
        for oracle in oracles:
            oracle.close()  # 所有查询完成或中断后释放 SP 求解器。


**3．统一的顶点切割循环。** LP 使用零径向容差，SOCP 使用同一 $\tau$。纯 SOCP 从总负荷单纯形出发，混合方法从该方案的线性外域出发。

维护已核验内点凸包 $I$ 和有效割外域 $U$；满足 $(1-\tau)U\subseteq I\subseteq D_{SOC}\subseteq U$ 后结束。这个条件不读取 AC 标签。固定背景负荷在内点构造时保持不变。

`halfspaces` 使用 $a^T p+b\leq 0$，SP 割使用 $c+d^T p\geq0$，加入外域时必须翻转符号并统一法向量长度。$\tau$ 是相对于固定背景下参数原点的径向精度，不是 FR 或 AC 电流等式误差。

In [7]:
def cut_region(oracle, initial, tolerance):  # 同一套固定方案顶点算法用于 LP 和 SOCP。
    equations = halfspaces(initial)  # 外域 U 的半空间约束 a·p+b≤0。
    inner, history = np.zeros((1,3)), []  # 从参数原点开始维护内点；当前算例的固定背景在原点可行。

    def check(point):  # 一次 SP 调用同时提供外割和经过原始约束核验的内点。
        nonlocal inner, equations  # 更新本方案当前的内点集合与外域半空间。
        eta, cut, witness, _ = oracle.solve(point)  # SOCP 内点是 witness，不能将含辅助违反量 eta 的查询解直接认证。
        inner = np.vstack([inner, witness])  # 相同方案的可行点凸包是有效内域 I。
        history.append(dict(p=point.copy(), eta=eta, cut=cut, witness=witness.copy()))  # 只记录此次实际查询，绘图不额外生成求解点。
        if cut is not None:  # 有可靠分离证书才收缩外域。
            equations = np.vstack([equations, -cut[[1,2,3,0]]/np.linalg.norm(cut[1:])])  # 将 c+b·p≥0 转为单位法向量的 -b·p-c≤0。
        return cut

    try:
        lengths = [np.min(-equations[equations[:,i]>1e-8,3]  # 由 a_i*p_i+b≤0 求各正坐标轴与初始外域的截距。
                          /equations[equations[:,i]>1e-8,i]) for i in range(3)]
        for point in .01*np.diag(lengths):  # 先查询三个轴截距的 1%，建立有三维内部的内点集合。
            check(point)
        interior = inner.mean(axis=0)  # 可行轴向内点的均值，是半空间求交所需的严格内部点。
        outer = HalfspaceIntersection(equations, interior).intersections  # 由当前有效外割重新求真正的外域顶点。
        for _ in range(2000):  # 只有下面的包含判据满足才返回，超限显式报未收敛。
            inside = halfspaces(inner)  # 内点凸包 I 的支持半空间；不是不同方案点的混合凸包。
            gap = np.max((1-tolerance)*outer@inside[:,:3].T+inside[:,3], axis=1)  # 每个径向内缩外顶点相对 I 各面的最大距离违反量。
            if gap.max() <= 1e-8:  # 所有 (1-tau)*U 顶点均在 I 内，由凸性得到整个包含关系。
                return dict(initial=initial, inner=polytope_vertices(inner), outer=outer,  # 只保存内域真正的凸包顶点，SP 历史仍完整保留。
                            history=history, calls=oracle.calls)
            point = np.maximum(outer[int(np.argmax(gap))], 0.)  # 优先检查内外差距最大的外顶点，并去掉负坐标舍入误差。
            point[point<1e-9] = 0.  # 把几何计算产生的极小轴向残差恢复为零。
            cut = check(point)  # 通过 SP 扩大内域、获得外割，或同时得到二者。
            if cut is None and np.any(inner[-1] < (1-tolerance)*point-1e-8):  # 若边界附近无可靠割且内点仍不足以满足径向精度，向内再查询一次。
                cut = check((1-.5*tolerance)*point)  # 查询半个径向容差以内的点，以获得足够靠近边界的可行证书。
            if cut is not None:  # 有可靠分离证书才收缩外域。
                outer = HalfspaceIntersection(equations, interior).intersections  # 由当前有效外割重新求真正的外域顶点。
        raise RuntimeError("Region did not reach its stopping tolerance")
    finally:
        oracle.close()  # 单方案构域拥有该求解器的完整生命周期。


**4．三种构域方法。** 每档预算分别重新求解。混合方法包含自己的线性阶段，计时不借用纯线性结果。所有方案上的外域最后取并集。

In [8]:
def compute_regions(plans, budget, method, tolerance):  # 预算内方案已筛选；输出逐方案区域及本方法完整构域耗时。
    start = perf_counter()  # 建模前开始计时，包含 LP/SOCP 模型构建与全部几何操作。
    lp_seconds, lp_calls, lp_cuts = 0., 0, 0  # 纯 SOCP 没有线性阶段，其对应统计为零。
    if method in ('linear','hybrid'):  # 混合方法独立执行自己的线性阶段，不能借用另一方法已完成的结果。
        if len(plans)==1:  # 单方案无需 MP 选择，直接用 LP 顶点切割求域。
            records = [dict(design=0, **cut_region(LinearSP(plans[0]), simplex(plans[0]), 0.))]  # LP 采用零径向容差；不引入 SOCP 的近似停止阈值。
        else:
            records = linear_region(plans, budget)  # 多方案使用 MP/SP 加认证并集覆盖的 LP 构域流程。
        lp_seconds = perf_counter()-start  # 混合方法的预处理时间包含在总时间中。
        lp_calls = sum(r['calls'] for r in records)  # 统计线性阶段真实 SP 调用次数。
        lp_cuts = sum(h['cut'] is not None for r in records for h in r['history'])  # 只统计真正加入的有效外割。
    if method in ('socp','hybrid'):  # 两种 SOCP 方法使用相同模型和相同径向停止精度。
        seeds = ([r['outer'] for r in records] if method=='hybrid'  # 混合方法从每个方案各自的线性外域出发。
                 else [simplex(d) for d in plans])  # 纯 SOCP 从总负荷单纯形出发。
        records = [dict(design=i, **cut_region(SOCPSP(d), initial, tolerance))  # 逐方案收紧外域；离散方案之间最终取并集。
                   for i,(d,initial) in enumerate(zip(plans,seeds))]
    timing = dict(region_seconds=perf_counter()-start, linear_stage_seconds=lp_seconds,  # 总构域时间已包含混合方法的线性阶段，汇总时不再加一次。
                  lp_calls=lp_calls, lp_cuts=lp_cuts)
    return records, timing


**5．独立 AC 参考与评价。** 完整 AC 保留 $P^2+Q^2=v\ell$，按费用从低到高检查方案。一个点首次得到可行证书时，其费用就是最小可行投资；较便宜方案存在未确定状态时必须继续核实，不能发布错误的最小费用。

AC 每个点只保存一个最小投资，多档预算直接查询，不重复扫描。AC 各预算耗时为本次顺序扫描处理完该预算全部方案的累计实际时间。三种构域方法的总时间包含建模、求解、几何处理和网格判定；公共网格准备、许可证启动及绘图不计。

AC 可行性由独立的等式模型确认；未确定点只有在找到同价或更低价的可行方案时，才不影响该点的最小可行投资。未解决的更便宜方案会阻止输出，避免将求解失败写成不可行标签。

In [9]:
def ac_reference(plans, points, budgets):  # plans 必须按费用升序排列，Network 已保证这一顺序。
    minimum = np.full(len(points), np.inf)  # 尚未找到 AC 可行方案的点暂记为无限费用。
    unknown = np.full(len(points), np.inf)  # 记录未确定方案的最低费用，防止误报最小可行投资。
    times, checks = [], 0  # 保存逐方案累计时间及实际执行的点—方案检查次数。
    environment = gp.Env(empty=True)  # 为可能需要的非凸 AC 核验建立静默求解环境。
    environment.setParam('OutputFlag', 0)
    environment.start()  # 许可证启动在 AC 扫描计时之前完成。
    start = perf_counter()  # 计时起点；在方法网格判定段中另起计时，用于构成总时间。
    try:
        for i,design in enumerate(plans):  # 从最低投资方案开始逐一检查。
            active = np.flatnonzero(np.isinf(minimum))  # 已经得到更便宜可行方案的点无需再检查高价方案。
            oracle = ACPowerFlow(design)  # 仅读取该方案网架，独立建立完整 AC 递推。
            try:
                status = oracle.scan(points[active])  # 批量 AC 三态判定：1 可行、-1 已证不可行、0 未确定。
                checks += len(active)  # 记录实际扫描工作量，不乘以预算数重复统计。
                for j in np.flatnonzero(status==0):  # 仅不动点尚未判定的点需要显式非凸 AC 求解。
                    status[j] = oracle.global_status(points[active[j]], environment)  # 超时或无证书仍是 0，不能当作不可行。
                minimum[active[status==1]] = design.cost  # 费用递增扫描下第一次可行的费用，是候选最小投资。
                unknown[active[status==0]] = np.minimum(unknown[active[status==0]], design.cost)  # 保留未判定的较低费用，最后检查它是否影响最小投资结论。
            finally:
                oracle.close()  # 结束该方案时释放可能创建的非凸模型。
            times.append(perf_counter()-start)  # 包含此前全部方案的累计扫描时间。
            print(f"AC 方案 {i+1}/{len(plans)}：{times[-1]:.1f} s", flush=True)
        if np.any(unknown < minimum):  # 若仍有更便宜的未确定方案，最小投资或不可行性都未被证实。
            raise RuntimeError("Unresolved AC feasibility at a cheaper design")
    finally:
        environment.dispose()  # 全部方案处理完后释放共享求解环境。
    costs = np.array([d.cost for d in plans])  # 用方案费用确定每档预算应计到哪一个扫描时刻。
    timings = [dict(region_seconds=times[np.flatnonzero(costs<=b)[-1]], evaluation_seconds=0.)  # AC 各预算耗时是同一次共享扫描的累计值，不是多次独立扫描。
               for b in budgets]
    return minimum, timings, checks


def evaluate(network, designs, budgets, divisions, tolerance):  # 统一评价箱和等体积网格，保证不同方法使用相同体积口径。
    # 评价箱覆盖所有允许方案的线性域，扩容后的区域不会被旧坐标范围截断。
    intercepts = []  # 收集所有允许方案的三个 LP 轴截距，用来包住扩容后的区域。
    for design in designs:
        e = BranchEquations(design)  # 读取固定背景下的无损线性约束 c+F*p≥0。
        intercepts.append([min(design.power_limit, np.min(-e.linear_c[e.linear_F[:,i]<0]  # 沿第 i 轴增载时，F_i<0 的运行限值给出 p_i≤-c/F_i。
                            /e.linear_F[e.linear_F[:,i]<0,i])) for i in range(3)])
    bounds = np.ceil(np.max(intercepts,axis=0)/10)*10  # 取所有方案截距最大值，并向上整到 10 kW，避免截断规划域。
    grid = np.indices((divisions,)*3, dtype=np.int32).reshape(3,-1).T  # 生成三维单元编号，每行对应一个体素。
    points = (grid+.5)*(bounds/divisions)  # 使用单元中心代表体素；各体素体积相同。
    selected = np.flatnonzero(points.sum(axis=1)<=max(d.power_limit for d in designs))  # 总负荷超出共同有效上界的点直接在所有区域之外。
    points = points[selected]  # 仅对评价箱中尚可能可行的点进行实际判定。
    masks = np.zeros((4,len(budgets),divisions**3), dtype=bool)  # 方法、预算、扁平网格三轴；跳过的点保持域外。
    regions = {m:[] for m in METHODS if m!='ac'}  # 三种切割方法保存几何，AC 用独立网格参考而不伪造凸包。
    timings = {m:[] for m in METHODS}  # 每个方法、每个预算对应一条实测计时记录。
    startup = gp.Model(); startup.dispose()  # 预先完成公共许可证启动，不把启动延迟只计给第一个方法。
    with threadpool_limits(limits=1):  # 矩阵运算统一单线程，减少方法比较时线程配置的差异。
        for j,budget in enumerate(budgets):  # 同一预算的三种切割方法使用完全相同的允许方案集合。
            plans = [d for d in designs if d.cost<=budget]  # 实现规划预算约束；各方案单独认证后取并集。
            for method in ('linear','socp','hybrid'):  # 每种方法重新建模并求解，混合方法包含自己的线性阶段。
                records, timing = compute_regions(plans,budget,method,tolerance)  # 计算该预算下的逐方案切割域。
                start = perf_counter()  # 计时起点；在方法网格判定段中另起计时，用于构成总时间。
                masks[METHODS.index(method),j,selected] = region_membership(  # 计算外域并集对同一网格的成员标签。
                    points,[np.asarray(r['outer']) for r in records if len(r['outer'])])  # 空方案域不贡献任何可行点；不能将不同方案合成一个凸包。
                timing['evaluation_seconds'] = perf_counter()-start  # 单独记录网格归属判定时间，再与构域时间相加。
                regions[method].append(records)
                timings[method].append(timing)
                print(f"预算 {budget:g}, {method}: {timing['region_seconds']+timing['evaluation_seconds']:.3f} s",flush=True)
        minimum, timings['ac'], checks = ac_reference(designs,points,budgets)  # 全部预算共享一遍按费用排序的独立 AC 扫描。
    ac_cost = np.full(divisions**3,np.inf)  # 恢复完整评价箱，之前排除的点保持不可行。
    ac_cost[selected] = minimum  # 每个网格点只保存一个最小可行投资。
    for j,budget in enumerate(budgets):  # 同一预算的三种切割方法使用完全相同的允许方案集合。
        masks[2,j] = np.isfinite(ac_cost)&(ac_cost<=budget)  # 存在预算内可行方案才属于 AC 规划域；inf≤inf 不能误判可行。
    metadata = dict(network=network.name, planning=PLANNING, cost_unit=network.cost_unit,  # 保存 Notebook 的运行设置和 Network 提供的物理语义。
        load_nodes=list(designs[0].load_nodes),bounds=bounds.tolist(),divisions=divisions,
        budgets=[None if np.isinf(b) else b for b in budgets],radial_tolerance=tolerance,  # 无限预算在 JSON 中用 null 表示。
        sample_count=len(points),timings=timings,point_design_checks=checks,
        designs=[dict(x=np.asarray(d.x).tolist(),cost=d.cost) for d in designs],  # 方案向量和费用只保存一张表，几何记录只引用编号。
        hashes={p:sha256(Path(p).read_bytes()).hexdigest() for p in  # 记录本次计算实际使用的模型和网架源码指纹。
                ('model.py','vertify.py','region.py',*network.sources)})
    metadata['hashes']['main.ipynb'] = sha256('\n'.join(c.source for c in nbformat.read('main.ipynb',as_version=4).cells).encode()).hexdigest()  # Notebook 指纹只含源单元，排除执行后变化的输出。
    result = BenchmarkResult(masks.reshape((4,len(budgets))+(divisions,)*3),metadata,regions,  # 还原三个空间轴，以同一数据驱动指标、存盘及绘图。
                             ac_cost.reshape((divisions,)*3))
    return result


**6．执行实验。** 上面的函数定义就是本实验使用的全部流程。重新运行此单元从头计算，不暗中续用另一次实验的求解结果。

In [10]:
if COMPACT_REVIEW:
    output = Path('results')/CASE/'compact_review'
    output.mkdir(parents=True,exist_ok=True)
    if RECOMPUTE:
        with threadpool_limits(limits=1):
            review = review_compact(network,budgets)
        np.savez_compressed(output/'review.npz',record=json.dumps(review,default=lambda x:x.tolist(),ensure_ascii=False))
    else:
        with np.load(output/'review.npz',allow_pickle=False) as data:
            review = json.loads(str(data['record']))
else:
    if RECOMPUTE:  # 显式重新计算时，从当前源码和网架开始完整实验。
        validation = validate_power_flow(designs[0]) if CASE=='case33bw' else None  # 33 节点先做独立节点导纳矩阵潮流交叉核验。
        result = evaluate(network,designs,budgets,DIVISIONS,RADIAL_TOLERANCE)  # 执行切割、独立 AC、网格成员判定及计时。
        result.metadata['validation'] = validation  # 将交叉核验依据并入本次唯一结果记录。
        result.save(OUTPUT)  # 整次实验只在此保存一次原始结果。
    else:
        result = BenchmarkResult.load(OUTPUT)  # 明确读取此前结果，不把旧时间标成当前重新求解时间。

Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2685996


Academic license 2685996 - for non-commercial use only - registered to 20___@mail.scut.edu.cn


linear: 124 个投资查询、48 个边界查询完成


linear: 联合割查询 36 次，全部 17 条割完成 16 方案连续域核验


socp: 124 个投资查询、48 个边界查询完成


socp: 联合割查询 36 次，全部 51 条割完成 16 方案连续域核验


**7．FR、MR 与求解时间。** $FR=\mathrm{Vol}(\widehat D\setminus D_{AC})/\mathrm{Vol}(\widehat D)$，$MR=\mathrm{Vol}(D_{AC}\setminus\widehat D)/\mathrm{Vol}(D_{AC})$。均在同一网格上计算，网格零误差不代表连续误差严格为零。

In [11]:
if COMPACT_REVIEW:
    compact_summary = pd.DataFrame(compact_review_summary(review))
    display(compact_summary.T)
    # 下列断言属于本次模型等价性实验，不进入通用求解器。
    assert (compact_summary.minimum_cost_mismatches==0).all()
    assert (compact_summary.budget_membership_mismatches==0).all()
    assert (compact_summary.joint_cost_mismatches==0).all()
    assert (compact_summary.max_radius_difference_kw<.002).all()
    assert (compact_summary.joint_max_radius_difference_kw<.002).all()
    assert (compact_summary.joint_max_gap_kw<=review['radial_gap_kw']+1e-5).all()
    assert (compact_summary.minimum_cut_margin>=-1e-7).all()
else:
    summary = pd.DataFrame(result.summary)  # FR/MR 从保存的基础标签现场推导，不维护第二份指标文件。
    summary['method'] = summary['method'].map(dict(zip(METHODS,METHOD_NAMES)))  # 仅替换显示名称，不修改方法轴和原始记录。
    display(summary[['budget','method','fr_percent','mr_percent','total_seconds']].rename(columns={
        'budget':f'预算 ({network.cost_unit})','method':'方法','fr_percent':'FR (%)',
        'mr_percent':'MR (%)','total_seconds':'总计算时间 (s)'}).style.format(precision=5))

,0,1
method,linear,socp
point_count,124,124
minimum_cost_mismatches,0,0
budget_membership_mismatches,0,0
radial_queries,48,48
max_radius_difference_kw,0.000001,0.000077
joint_queries,36,36
joint_cost_mismatches,0,0
joint_max_radius_difference_kw,0.001,0.001
joint_max_gap_kw,0.001,0.001001


**8．交互图与切割回放。** 蓝色为与 AC 重合，红色遗漏，黄色多余；同预算四幅图同步旋转。回放从已保存的 SP 查询记录重建固定方案的内外域，不调用求解器。

In [12]:
if not COMPACT_REVIEW:
    page = save_method_comparison(result,OUTPUT)  # 图形与 FR/MR 消费同一网格标签，红色遗漏、黄色多余。
    replay = save_replay(result,OUTPUT)  # 从真实 SP 历史重建切割过程，不额外求解表面网格点。
    display(IFrame(src=page.as_posix(),width='100%',height=1220))
    display(IFrame(src=replay.as_posix(),width='100%',height=740))